# 01 · Esqueleto: baseline do aluno

Fatia fina de ponta a ponta, do dado até uma métrica. **O baseline é descartável**: serve para provar que o caminho snapshot → tabela de modelagem → `Pipeline` → AUC-ROC funciona. A AUC-ROC daqui **não é resultado do projeto**; as unidades de pré-processamento, validação e modelagem a substituem.

In [1]:
import sys
from pathlib import Path

# Os notebooks chamam o código de `src/`; a raiz do repositório entra no caminho de importação.
sys.path.insert(0, str(Path.cwd().parent))

from src.modeling.baseline import avaliar_baselines, montar_tabela_aluno
from src.preprocessing.loader import carregar_tabela

## Tabela de modelagem

Une `fact_alunos` aos atributos de `dim_municipio`. O alvo `nao_alfabetizado` tem a classe positiva = não alfabetizado (o risco). `presenca`, `preenchimento_caderno` e `proficiencia` ficam de fora: caderno em branco implica `alfabetizado` falso, e a proficiência define o rótulo (ver `data/README.md`).

In [2]:
alunos = carregar_tabela("fact_alunos")
municipios = carregar_tabela("dim_municipio")
tabela = montar_tabela_aluno(alunos, municipios)
print(tabela.shape)
tabela.head()

(3867999, 10)


,rede,caderno,sigla_uf,nome_regiao,capital_uf,idhm,idhm_educacao,idhm_renda,idhm_longevidade,nao_alfabetizado
0,2,1,DF,Centro-Oeste,1.0,0.824,0.742,0.863,0.873,1
1,2,1,DF,Centro-Oeste,1.0,0.824,0.742,0.863,0.873,1
2,2,1,DF,Centro-Oeste,1.0,0.824,0.742,0.863,0.873,1
3,2,1,DF,Centro-Oeste,1.0,0.824,0.742,0.863,0.873,1
4,2,1,DF,Centro-Oeste,1.0,0.824,0.742,0.863,0.873,1


## Baselines

`DummyClassifier` (prior) e Regressão Logística, ambos dentro de um `Pipeline` com imputação, escalonamento e One-Hot ajustados só no treino. Separação 80/20 estratificada pelo alvo, com semente fixa.

In [3]:
resultado = avaliar_baselines(tabela)
resultado

{'linhas_treino': 3094399.0,
 'linhas_validacao': 773600.0,
 'proporcao_positiva': 0.48693213209207137,
 'auc_roc_dummy': 0.5,
 'auc_roc_regressao_logistica': 0.6272736127683466}

## Leitura

- O `DummyClassifier` dá AUC-ROC 0,5, como esperado para um classificador que só responde a proporção da classe.
- A Regressão Logística passa de 0,5, mas o valor é **otimista**: o `train_test_split` não separa por município, então o mesmo município aparece no treino e na validação, e os atributos municipais o identificam. A separação por município é da unidade de validação.